# Respiratory Sound Classification — ICBHI 2017 (Recovery Run)

End-to-end Google Colab notebook for the **AST + SAM** baseline.  
Target: beat the reference paper (Se 68.31%, Sp 67.89%, Score 68.10%).

What changed vs. the previous run
- AST is loaded with its **native** input shape (16 kHz, 128 mels, 1024 frames) so the pretrained positional embeddings are preserved.
- AudioSet mean/std normalization is done by `ASTFeatureExtractor`, not by us.
- A patient-level **validation split** is carved out of `train.csv`. The test set is held out completely until the very end.
- **Two-stage training**: head-only warmup → unfreeze backbone with layer-wise LR decay.
- Class imbalance: **square-root sampling** (less aggressive than inverse) + class-balanced **Focal Loss** + label smoothing.
- Augmentation: waveform aug + SpecAugment + Mixup.
- SAM optimizer (rho=0.05), gradient clipping, linear warmup → cosine LR schedule, AMP, early stopping.

Runtime: pick **T4 GPU** in `Runtime → Change runtime type`.

## 1. Mount Drive & locate the dataset

We expect a folder `MyDrive/respiratory_project/` on your Drive that contains  
`ICBHI_final_database.zip` (or the already-extracted folder).  
Checkpoints and logs will be written under the same folder so you don't lose anything on a Colab restart.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob

DRIVE_BASE = "/content/drive/MyDrive/respiratory_project"
RAW_DIR    = "/content/data/raw"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/logs", exist_ok=True)

# Unzip the dataset on the first run (~1 min on Colab SSD).
zip_path = f"{DRIVE_BASE}/ICBHI_final_database.zip"
extract_check = f"{RAW_DIR}/ICBHI_final_database/filename_differences.txt"

if not os.path.exists(extract_check):
    print("Unzipping ICBHI_final_database.zip ...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_DIR)
    print("done.")
else:
    print("Dataset already extracted.")

wavs = glob.glob(f"{RAW_DIR}/ICBHI_final_database/*.wav")
print(f"WAV files found: {len(wavs)}")  # expected: 920
assert len(wavs) == 920, "Expected 920 wav files in the ICBHI_final_database folder."

## 2. Clone the GitHub repo (or pull latest)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/Younes-Abbes/respiratory-classification.git"
REPO_DIR = "/content/respiratory-classification"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())
print(subprocess.check_output(["git", "log", "--oneline", "-1"]).decode().strip())

## 3. Install dependencies & verify GPU

In [ ]:
%pip install -q -r requirements_colab.txt

In [ ]:
import torch, os

print("torch       :", torch.__version__)
print("cuda avail. :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device      :", torch.cuda.get_device_name(0))
    print("memory total:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

assert torch.cuda.is_available(), "No GPU detected — switch runtime to T4 GPU."

# WandB is optional, keep it off to avoid prompts during long training.
os.environ["WANDB_DISABLED"] = "true"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 4. Wire the dataset into the repo & build splits

We symlink the unzipped ICBHI folder into `data/raw/` (the repo scripts expect this layout) and regenerate the patient-level splits.

In [ ]:
import os, glob, shutil
from pathlib import Path

REPO_DIR = "/content/respiratory-classification"
os.chdir(REPO_DIR)

# Flatten dataset into data/raw/ (no subfolder).
RAW = Path("data/raw")
RAW.mkdir(parents=True, exist_ok=True)
src_dir = Path("/content/data/raw/ICBHI_final_database")
if src_dir.exists():
    files = list(src_dir.glob("*.wav")) + list(src_dir.glob("*.txt"))
    moved = 0
    for f in files:
        dst = RAW / f.name
        if not dst.exists():
            try:
                os.symlink(f, dst)
            except OSError:
                shutil.copy(f, dst)
            moved += 1
    print(f"Linked {moved} files into data/raw/")
else:
    print(f"WARN: source dir not found: {src_dir}")

wavs = list(RAW.glob("*.wav"))
print(f"WAV files in data/raw : {len(wavs)} (expected 920)")
assert len(wavs) == 920

In [ ]:
import os
from pathlib import Path

# Rebuild patient-level splits if missing.
if not Path("data/splits/train.csv").exists():
    !python scripts/prepare_data.py
else:
    print("data/splits/train.csv already exists - skipping.")

import pandas as pd
train_df = pd.read_csv("data/splits/train.csv")
test_df  = pd.read_csv("data/splits/test.csv")
print("Train cycles :", len(train_df), "| patients:", train_df["patient_id"].nunique())
print("Test  cycles :", len(test_df),  "| patients:", test_df["patient_id"].nunique())
print("\nClass distribution (train):")
print(train_df["label_name"].value_counts())
print("\nClass distribution (test):")
print(test_df["label_name"].value_counts())

## 5. Sanity check — AST forward pass

If this cell runs without warnings about positional-embedding mismatch, the AST is loaded correctly.

In [ ]:
import sys
sys.path.insert(0, "/content/respiratory-classification")

import torch, yaml, numpy as np
from src.model import load_model
from src.dataset import ICBHIASTDataset
from transformers import ASTFeatureExtractor

with open("configs/baseline.yaml") as f:
    config = yaml.safe_load(f)

print("Config (key values):")
for k in ["use_pretrained", "freeze_backbone", "sample_rate", "ast_max_length",
          "batch_size", "epochs", "warmup_epochs", "loss", "rho",
          "use_mixup", "use_specaugment", "sampler"]:
    print(f"  {k:18s}: {config.get(k)}")

fe = ASTFeatureExtractor.from_pretrained(config["backbone_name"])
ds = ICBHIASTDataset("data/splits/train.csv", config, feature_extractor=fe, augment=False)

spec, label = ds[0]
print("\nFeature extractor output (single sample):")
print("  spec shape :", tuple(spec.shape), "(expected (1024, 128))")
print("  label      :", int(label))
print("  spec range :", float(spec.min()), "to", float(spec.max()))

model = load_model(use_pretrained=True, freeze_backbone=False,
                   head_hidden_dim=config["head_hidden_dim"],
                   head_dropout=config["head_dropout"]).cuda()
with torch.no_grad():
    out = model(spec.unsqueeze(0).cuda())
print("\nModel forward:")
print("  logits shape:", tuple(out.shape), "(expected (1, 4))")
print("  logits      :", out.cpu().numpy())

## 6. Dry run (2 epochs × 5 batches)

A quick sanity check that:
- the training loop runs,
- mixup + SpecAugment + SAM all play nicely together,
- a checkpoint is saved.

If this completes in under a minute on T4, we are good to launch the real run.

In [ ]:
!python -u -m src.train --dry-run \
    --checkpoint-dir /content/respiratory-classification/checkpoints \
    --log-dir /content/respiratory-classification/logs

## 7. Clear old checkpoints from the failed run

The previous checkpoint stored Fallback-CNN weights and an `epoch=0`. We delete it so the AST run starts from scratch and writes a fresh `best_model.pt`.

In [ ]:
import os, shutil
from pathlib import Path

LOCAL_CKPT = Path("/content/respiratory-classification/checkpoints")
LOCAL_LOGS = Path("/content/respiratory-classification/logs")
DRIVE_CKPT = Path("/content/drive/MyDrive/respiratory_project/checkpoints")
DRIVE_LOGS = Path("/content/drive/MyDrive/respiratory_project/logs")

for d in [LOCAL_CKPT, LOCAL_LOGS, DRIVE_CKPT, DRIVE_LOGS]:
    d.mkdir(parents=True, exist_ok=True)

# Remove local dry-run artifacts so the real run starts clean.
for f in LOCAL_CKPT.glob("*.pt"):
    print("removing local ckpt:", f); f.unlink()
for f in LOCAL_LOGS.glob("*.csv"):
    print("removing local log:", f); f.unlink()

# Remove old broken Drive checkpoints + stale metrics from the previous failed run.
# The previous best_model.pt contained the fallback-CNN; if we leave it, evaluate_quick
# would try to load it and report bogus numbers.
for f in DRIVE_CKPT.glob("*.pt"):
    print("removing drive ckpt:", f); f.unlink()
for f in DRIVE_LOGS.glob("metrics*.csv"):
    print("removing drive metrics:", f); f.unlink()

print("\nClean. Ready to launch full training.")

## 8. Full training run

Writes checkpoints **directly into Google Drive** so the run survives a Colab disconnect.  
On a T4 this takes roughly 25–35 minutes (40 epochs, batch size 16, AMP on).

If the run dies midway, just re-run this cell and pass `--resume` — training will pick up from `latest.pt`.

In [ ]:
!python -u -m src.train \
    --checkpoint-dir /content/drive/MyDrive/respiratory_project/checkpoints \
    --log-dir /content/drive/MyDrive/respiratory_project/logs

### 8b. Resume training if interrupted

Run this cell only if the previous one crashed before completion. It loads `latest.pt` and continues.

In [ ]:
# Uncomment to resume from the last saved checkpoint.
# !python -u -m src.train --resume \
#     --checkpoint-dir /content/drive/MyDrive/respiratory_project/checkpoints \
#     --log-dir /content/drive/MyDrive/respiratory_project/logs

## 9. Training curves

Plots train/val Score, Se and Sp from the metrics CSV written during training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

LOG_PATH = Path("/content/drive/MyDrive/respiratory_project/logs/metrics.csv")
m = pd.read_csv(LOG_PATH)
m = m[m["epoch"].astype(str).str.isdigit()]
m["epoch"] = m["epoch"].astype(int)
print(m.tail())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(m["epoch"], m["train_loss"], label="train_loss")
axes[0].set_title("Training loss"); axes[0].set_xlabel("epoch"); axes[0].grid(True)

axes[1].plot(m["epoch"], m["train_Score"], label="train Score")
axes[1].plot(m["epoch"], m["val_Score"],   label="val Score")
axes[1].axhline(0.6810, color="red", ls="--", lw=1, label="ref paper 0.6810")
axes[1].set_title("ICBHI Score"); axes[1].set_xlabel("epoch"); axes[1].grid(True); axes[1].legend()

axes[2].plot(m["epoch"], m["val_Se"], label="val Se")
axes[2].plot(m["epoch"], m["val_Sp"], label="val Sp")
axes[2].axhline(0.6831, color="red", ls="--", lw=1, label="ref Se 0.6831")
axes[2].set_title("Sensitivity / Specificity"); axes[2].set_xlabel("epoch"); axes[2].grid(True); axes[2].legend()

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/respiratory_project/logs/training_curves.png", dpi=150)
plt.show()

## 10. Final evaluation on the official test split

Loads `best_model.pt` from Drive and evaluates it on `data/splits/test.csv`. Produces the binary `Se / Sp / Score` metrics required by the ICBHI protocol and a 4-class confusion matrix.

In [ ]:
!python scripts/evaluate_quick.py \
    --checkpoint /content/drive/MyDrive/respiratory_project/checkpoints/best_model.pt \
    --config configs/baseline.yaml \
    --csv data/splits/test.csv \
    --logs-dir /content/drive/MyDrive/respiratory_project/logs

In [ ]:
from IPython.display import Image, display
display(Image("/content/drive/MyDrive/respiratory_project/logs/confusion_matrix_baseline.png"))

## 11. Stop / Go gates by epoch

Use this table to decide whether to keep going or change strategy. Each gate is checked against **val Score** (not test, never test).

| Epoch | Minimum val Score | What to do if you fail it |
|------|------|------|
| 5    | ≥ 0.55 | Verify the AST forward shape and the spec normalisation. |
| 10   | ≥ 0.60 | Lower `mixup_alpha` to 0.1 and reduce `spec_time_mask` to 64. |
| 20   | ≥ 0.64 | Set `loss: ce`, `label_smoothing: 0.1`, `sampler: none`. |
| 30   | ≥ 0.67 | Switch to `class_balanced_beta: 0.9` and increase `head_lr` to 2e-4. |
| 40   | ≥ 0.68 | You have beaten the reference paper. Run improvements (mixup → patch-mix, ensemble with EfficientNet). |

If you ever see **train Se ~ 0.9** and **train Sp ~ 0.3** like the previous run, the model is collapsing into a degenerate "predict abnormal" solution.  
Fixes (already applied in this config but listed as a safety net):
1. Set `sampler: sqrt` (not `inverse`).
2. Lower the focal-loss `class_balanced_beta` from 0.9999 to 0.99 (gentler reweighting).
3. Reduce `head_dropout` from 0.3 to 0.15 so the head doesn't have to over-correct.

## 12. Backup everything to Drive (final)

This is automatic if you trained directly to the Drive folder, but the cell makes sure nothing local is lost.

In [ ]:
import shutil
from pathlib import Path

LOCAL_CKPT = Path("/content/respiratory-classification/checkpoints")
LOCAL_LOGS = Path("/content/respiratory-classification/logs")
DRIVE_CKPT = Path("/content/drive/MyDrive/respiratory_project/checkpoints")
DRIVE_LOGS = Path("/content/drive/MyDrive/respiratory_project/logs")

for f in LOCAL_CKPT.glob("*.pt"):
    shutil.copy(f, DRIVE_CKPT / f.name); print("ckpt ->", f.name)
for f in LOCAL_LOGS.glob("*"):
    if f.is_file():
        shutil.copy(f, DRIVE_LOGS / f.name); print("log  ->", f.name)
print("\nAll saved. Safe to close.")